In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.44.2 datasets==2.19.0

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/AncientRusProject_V4"
TOKENIZER_DIR = f"{BASE_DIR}/ancient_rus_tokenizer"
DATA_FILE = f"{BASE_DIR}/ancient_rus_ready_for_bert.txt"
MODEL_DIR = f"{BASE_DIR}/mini_bert_ancient_rus"

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    BertConfig, BertForMaskedLM, BertTokenizerFast,
    DataCollatorForWholeWordMask, Trainer, TrainingArguments
)

In [ ]:
dataset = load_dataset("text", data_files={"train": DATA_FILE})
tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_DIR)

In [ ]:
split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=False)

In [ ]:
tokenized_datasets = split_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

In [ ]:
def group_texts(examples):
    block_size = 256
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

In [ ]:
config = BertConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,          # 🔥 Увеличили в 2 раза (было 256)
    num_hidden_layers=6,      # 🔥 Добавили слоев (было 4)
    num_attention_heads=8,    # 🔥 Увеличили в 2 раза (было 4)
    intermediate_size=2048,   # 🔥 Увеличили (было 1024)
    max_position_embeddings=512,
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
model = BertForMaskedLM(config)
print(f"🧠 Параметры модели: {model.num_parameters():,} (Оптимизировано для быстрого обучения)")

🧠 Параметры модели: 34,833,202 (Оптимизировано для быстрого обучения)


In [ ]:
data_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
from transformers import TrainerCallback

In [ ]:
class SmartPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        # Проверяем каждые 500 шагов
        if state.global_step % 500 == 0:
            print(f"\n🔮 --- ПРОВЕРКА НА ШАГЕ {state.global_step} ---")

            # Примеры для ВСЕХ 6 категорий нашего мега-корпуса
            samples = [
                # 1. Церковь (Ожидаем: сына / сн҃а)
                "[CTX_CHURCH] во имѧ ѿц҃а и [MASK] и ст҃го дх҃а",

                # 2. Быт / Грамоты (Ожидаем: брату / господину / марии)
                "[CTX_DAILY] поклоно ѿ онѳима ко [MASK]",

                # 3. Закон / Суд (Ожидаем: имати / брати / судити)
                "[CTX_LEGAL] а посулов бояром не [MASK]",

                # 4. Литература / Летописи (Ожидаем: словесы)
                "[CTX_LIT] не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть",

                # 5. Эпос / Былины (Ожидаем: молодец / богатырь)
                "[CTX_EPIC] гой еси ты добрый [MASK]",

                # 6. Наука / Медицина (Ожидаем: зеліе / траву / воду)
                "[CTX_SCIENCE] а ѿ тоя болезни дай ему пити [MASK]"
            ]

            device = kwargs['model'].device
            kwargs['model'].eval() # Переключаем в режим оценки, чтобы не портить градиенты

            with torch.no_grad():
                for text in samples:
                    inputs = tokenizer(text, return_tensors="pt").to(device)
                    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

                    if len(mask_token_index) > 0:
                        outputs = kwargs['model'](**inputs)
                        logits = outputs.logits

                        # Берем топ-3 предсказания для маски
                        mask_token_logits = logits[0, mask_token_index, :]
                        top_3_tokens = torch.topk(mask_token_logits, 3, dim=1).indices[0].tolist()

                        decoded = [tokenizer.decode([token]) for token in top_3_tokens]

                        # Красивое форматирование вывода
                        category = text.split(']')[0] + ']'
                        clean_text = text.replace(category, '').strip()
                        print(f"📝 {category:<13} | {clean_text}  ->  {decoded}")

            kwargs['model'].train() # Возвращаем модель в режим обучения
            print("----------------------------------------------\n")

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    overwrite_output_dir=True,
    num_train_epochs=15,
    per_device_train_batch_size=64, # Если не влезет, ставь 64
    gradient_accumulation_steps=2,

    evaluation_strategy="steps",
    eval_steps=400,
    save_steps=400,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=True,
    fp16=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,


    learning_rate=5e-4,          # Увеличили скорость обучения в 10 раз!
    lr_scheduler_type="cosine",
    warmup_steps=1000,           # Даем больше времени на разогрев с новым мощным LR
    weight_decay=0.01,           # Поможет избежать переобучения
    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    callbacks=[SmartPrinterCallback()]
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
400,7.266500,7.339069
800,6.912600,6.808089
1200,6.202800,6.005888
1600,5.373900,5.231227
2000,4.874400,4.775397
2400,4.525300,4.418006
2800,4.221500,4.111028
3200,4.000600,3.893450
3600,3.820100,3.731257
4000,3.688400,3.608306



🔮 --- ПРОВЕРКА НА ШАГЕ 500 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['и', '.', ',']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['.', 'а', 'и']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['а', '.', ',']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  [',', '.', 'и']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  ['.', ',', 'и']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]  ->  [',', 'и', '.']
----------------------------------------------


🔮 --- ПРОВЕРКА НА ШАГЕ 1000 ---
📝 [CTX_CHURCH]  | во имѧ ѿц҃а и [MASK] и ст҃го дх҃а  ->  ['и', 'на', 'ст҃го']
📝 [CTX_DAILY]   | поклоно ѿ онѳима ко [MASK]  ->  ['поклоно', 'ѿ', '.']
📝 [CTX_LEGAL]   | а посулов бояром не [MASK]  ->  ['.', ';', ',']
📝 [CTX_LIT]     | не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть  ->  ['и', ',', '.']
📝 [CTX_EPIC]    | гой еси ты добрый [MASK]  ->  [',', '.', '!']
📝 [CTX_SCIENCE] | а ѿ тоя болезни дай ему пити [MASK]

There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=5415, training_loss=4.788353655983316, metrics={'train_runtime': 4318.7765, 'train_samples_per_second': 160.389, 'train_steps_per_second': 1.254, 'total_flos': 2.04376981890816e+16, 'train_loss': 4.788353655983316, 'epoch': 15.0})

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

('/content/drive/MyDrive/AncientRusProject_V4/mini_bert_ancient_rus/tokenizer_config.json',
 '/content/drive/MyDrive/AncientRusProject_V4/mini_bert_ancient_rus/special_tokens_map.json',
 '/content/drive/MyDrive/AncientRusProject_V4/mini_bert_ancient_rus/vocab.txt',
 '/content/drive/MyDrive/AncientRusProject_V4/mini_bert_ancient_rus/added_tokens.json',
 '/content/drive/MyDrive/AncientRusProject_V4/mini_bert_ancient_rus/tokenizer.json')

In [ ]:
import math

# Оценка на валидационной выборке
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])

print(f"📊 Результаты после 15 эпох:")
print(f"Финишый Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {perplexity:.2f}")

if perplexity < 20:
    print("🏆 Модель великолепно выучила структуру языка!")
elif perplexity < 50:
    print("📈 Хороший результат, модель понимает контекст.")
else:
    print("⚠️ Модели было сложно. Возможно, нужно больше данных или слоев.")

📊 Результаты после 15 эпох:
Финишый Loss: 3.4683
Perplexity: 32.08
📈 Хороший результат, модель понимает контекст.


In [ ]:
from transformers import pipeline

print("🔍 Загрузка обученной модели для финального теста...")
fill_mask = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    device=0,
)

# Тесты для каждой категории
final_tests = [
    {
        "category": "⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)",
        "text": "[CTX_CHURCH] Во имя отца и [MASK] и святаго духа."
    },
    {
        "category": "🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)",
        "text": "[CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ."
    },
    {
        "category": "⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)",
        "text": "[CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] ."
    },
    {
        "category": "📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)",
        "text": "[CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть."
    },
    {
        "category": "⚔️ [CTX_EPIC] (Ожидаем: молодец / богатырь / конь)",
        "text": "[CTX_EPIC] Гой еси ты добрый [MASK] , куда путь держишь?"
    },
    {
        "category": "🌿 [CTX_SCIENCE] (Ожидаем: зеліе / траву / воду)",
        "text": "[CTX_SCIENCE] А ѿ тоя болезни дай ему пити [MASK] , и тако исцелеет."
    }
]

print("\n" + "=" * 60)
print("🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)")
print("=" * 60)

for test in final_tests:
    print(f"\n🔹 {test['category']}")
    print(f"Текст: {test['text']}")
    results = fill_mask(test["text"])
    for i, res in enumerate(results[:3]):
        print(f"  {i+1}. {res['token_str']:<12} (Уверенность: {res['score']*100:.1f}%)")

🔍 Загрузка обученной модели для финального теста...

🏆 ФИНАЛЬНЫЙ ЭКЗАМЕН MINI-BERT (ВСЕ КАТЕГОРИИ)

🔹 ⛪️ [CTX_CHURCH] (Ожидаем: сына / отца / бога / духа)
Текст: [CTX_CHURCH] Во имя отца и [MASK] и святаго духа.
  1. сына         (Уверенность: 90.8%)
  2. отца         (Уверенность: 2.5%)
  3. матери       (Уверенность: 2.0%)

🔹 🏡 [CTX_DAILY] (Ожидаем: господину / брату / юрью)
Текст: [CTX_DAILY] Поклонъ ѿ бориса ко [MASK] съ бг҃омъ.
  1. носта        (Уверенность: 23.9%)
  2. брату        (Уверенность: 8.7%)
  3. гюргю        (Уверенность: 8.5%)

🔹 ⚖️ [CTX_LEGAL] (Ожидаем: винити / судити / имати / дати)
Текст: [CTX_LEGAL] Аже оубиеть моужь мужа, то мьстити брату, а посулов не [MASK] .
  1. надобѣ       (Уверенность: 38.5%)
  2. имати        (Уверенность: 33.2%)
  3. платити      (Уверенность: 10.1%)

🔹 📚 [CTX_LIT] (Ожидаем: словесы / дѣлы)
Текст: [CTX_LIT] Не лѣпо ли ны бяшетъ братие начяти старыми [MASK] трудную повѣсть.
  1. ,            (Уверенность: 10.4%)
  2. и            (Увере